# 04 — Eval pipeline

Scores the Stage 1 planner adapter on the **held-out** examples it never
trained on, and writes the results plus `.png` charts to Drive.

**`Runtime -> Run all` on a GPU runtime is the whole procedure.** Section 2
sets the runtime up; section 3 loads the adapter from Drive; sections 4-6
generate, score, and chart.

### What gets scored, and why it's layered

Execution alone would be almost no signal here. Of the 50 reference
snippets, only 4 run cleanly in a bare runtime — the rest need a display
(`tkinter`), the network (`requests`, `flask`), heavy deps, or CSV files
that aren't in the repo. Scoring only on execution would rest a 10-example
eval on ~0 examples.

So `src/eval/scoring.py` reports tiers, each with its own denominator:

| Tier | Question | Coverage |
| --- | --- | --- |
| `plan_well_formed` | does `validate_plan` accept it? | all examples |
| `plan_verb_sequence_match` | same verbs, same order as the reference? | all examples |
| `plan_exact_match` | byte-identical after whitespace normalization? | all examples |
| `code_parses` | does `ast.parse` accept it? | all examples (Stage 2) |
| `code_self_contained` | safe to `exec` here at all? | all examples (Stage 2) |
| `code_executes` | runs to completion without raising? | only where the *reference* also runs |

The last tier gates on the reference on purpose: if the reference itself
can't run here, a correct generation couldn't either, and scoring it would
punish the model for a missing CSV.

**Known limitation of the current split:** with `SPLIT_SEED = 0` none of the
4 executable examples land in the held-out set, so `code_executes` will
report *not attempted* rather than a rate. Re-seeding to get a better draw
would be choosing the split after seeing the answers, so the seed stays put
and the chart says "not attempted" honestly.

## 1. Config

`ADAPTER_DIR` must be the directory `02_stage1_finetune.ipynb` section 9
wrote. `SPLIT_SEED` and `EVAL_FRACTION` must match that notebook's values —
if they don't, this scores examples the model trained on.

In [ ]:
REPO_URL = "https://github.com/ashfordreyes/pythonllm.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/pythonllm"

STAGE1_DATA = f"{REPO_DIR}/data/stage1_planner/englishtopseudo.jsonl"
STAGE2_DATA = f"{REPO_DIR}/data/stage2_coder/pseudotopython.jsonl"

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/pythonllm_checkpoints"
ADAPTER_DIR = f"{DRIVE_CHECKPOINT_DIR}/stage1_planner"
RESULTS_DIR = f"{DRIVE_CHECKPOINT_DIR}/eval_stage1"

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 512

# Must match 02_stage1_finetune.ipynb, or this evaluates on training data.
SPLIT_SEED = 0
EVAL_FRACTION = 0.2

## 2. Setup

Same four steps as notebook 02: check the GPU, install, mount Drive, clone
the repo. Generation needs far less memory than training, so an L4 is
plenty here.

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.isdir(ADAPTER_DIR), (
    f"no adapter at {ADAPTER_DIR} — run 02_stage1_finetune.ipynb through "
    "section 9 first, or point ADAPTER_DIR at wherever you saved it"
)
print("Adapter:", ADAPTER_DIR)
print("Results will go to:", RESULTS_DIR)

In [ ]:
import pathlib
import sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present; pulling")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

for path in (STAGE1_DATA, STAGE2_DATA):
    assert pathlib.Path(path).exists(), f"missing {path} — the clone failed"

sys.path.insert(0, f"{REPO_DIR}/src")
print("OK")

## 3. Load the adapter

The tokenizer comes from `ADAPTER_DIR`, not from the hub — that is where the
DSL special tokens live, and an adapter loaded against a stock-vocab
tokenizer will spell `<PLAN>` out character by character.

4-bit quantization here is only to fit comfortably on a smaller GPU; nothing
about the adapter requires it. See `docs/colab_setup.md` §8.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

from dsl.schema import SPECIAL_TOKENS

ids = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)
assert all(i is not None and i != tokenizer.unk_token_id for i in ids), (
    f"tokenizer at {ADAPTER_DIR} is missing the DSL tokens: {dict(zip(SPECIAL_TOKENS, ids))}"
)
print("DSL token ids:", dict(zip(SPECIAL_TOKENS, ids)))

## 4. Generate plans for the held-out tasks

`splits.load_split` re-derives the same held-out indices notebook 02 used,
from the seed alone — there is no split file to keep in sync.

Greedy decoding (`do_sample=False`) so the numbers are reproducible.

In [ ]:
from splits import EVAL, load_split

eval_rows = load_split(STAGE1_DATA, EVAL, SPLIT_SEED, EVAL_FRACTION)
print(f"{len(eval_rows)} held-out examples")

SYSTEM_PROMPT = (
    "You are a planner that turns a task description into a pseudocode plan "
    "using the pythonllm DSL. Respond with only the plan, wrapped in "
    "<PLAN>...</PLAN> and made of <STEP> lines."
)


def generate_plan(english: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": english},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    new_tokens = out[0][encoded["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False).replace(
        tokenizer.eos_token, ""
    ).strip()


generations = []
for i, row in enumerate(eval_rows, start=1):
    plan = generate_plan(row["english"])
    generations.append({"english": row["english"], "reference": row["pseudocode"], "generated": plan})
    print(f"[{i}/{len(eval_rows)}] {row['english'][:70]}")

print("\n--- first generation ---")
print(generations[0]["generated"])

## 5. Score

Every tier is reported with its own denominator, so a tier that ran on 0
examples reads as *not attempted* rather than as 0%.

In [ ]:
import json

from eval.scoring import aggregate, score_plan

plan_scores = [score_plan(g["generated"], g["reference"]) for g in generations]
summary = aggregate(plan_scores)

for name, tier in summary["tiers"].items():
    if tier["rate"] is None:
        print(f"{name:28s} not attempted")
    else:
        print(f"{name:28s} {tier['passed']}/{tier['attempted']}  ({tier['rate']:.0%})")

if summary["plan_error_types"]:
    print("\nvalidation errors:", summary["plan_error_types"])

RESULTS_PATH = f"{RESULTS_DIR}/stage1_results.json"
with open(RESULTS_PATH, "w") as f:
    json.dump(
        {
            "split": {"seed": SPLIT_SEED, "eval_fraction": EVAL_FRACTION, "n": len(eval_rows)},
            "adapter": ADAPTER_DIR,
            "summary": summary,
            "generations": [
                dict(g, score=vars(s)) for g, s in zip(generations, plan_scores)
            ],
        },
        f,
        indent=2,
    )
print(f"\nWrote {RESULTS_PATH}")

## 6. Charts

Two PNGs into `RESULTS_DIR`: pass rate per tier, and which validation errors
the failures actually were. If the loss curve from notebook 02 was copied
next to the adapter, it's displayed here too so the run reads as one piece.

In [ ]:
from IPython.display import Image, display

from eval.plots import plot_plan_error_types, plot_score_breakdown

breakdown_png = plot_score_breakdown(summary, f"{RESULTS_DIR}/score_breakdown.png")
errors_png = plot_plan_error_types(summary, f"{RESULTS_DIR}/plan_error_types.png")

for png in (breakdown_png, errors_png):
    print(png)
    display(Image(str(png)))

loss_png = pathlib.Path(ADAPTER_DIR) / "loss_curve.png"
if loss_png.exists():
    display(Image(str(loss_png)))
else:
    print(f"(no loss curve at {loss_png} — notebook 02 section 9 copies it there)")

## 7. Stage 2 and end-to-end — *requires `03_stage2_finetune.ipynb`*

Not runnable yet: there is no coder adapter to load. The scoring side is
already built (`score_code`, `is_executable_example` in
`src/eval/scoring.py`), and the split helper hands back the *same* held-out
rows for `data/stage2_coder/pseudotopython.jsonl`, so end-to-end eval scores
Stage 1 and Stage 2 on the same tasks.

Once notebook 03 exists, this section becomes:

```python
from splits import EVAL, load_split
from eval.scoring import aggregate, is_executable_example, score_code, score_plan

stage2_rows = load_split(STAGE2_DATA, EVAL, SPLIT_SEED, EVAL_FRACTION)

# Stage 2 alone: reference pseudocode -> code
code_scores = [
    score_code(generate_code(r["pseudocode"]),
               attempt_execution=is_executable_example(r["python_code"]))
    for r in stage2_rows
]

# End to end: english -> generated plan -> code
e2e_scores = [
    score_code(generate_code(g["generated"]),
               attempt_execution=is_executable_example(r["python_code"]))
    for g, r in zip(generations, stage2_rows)
]

aggregate(plan_scores, code_scores)
```

`aggregate` already accepts the code tier; nothing here needs new scoring
code, only a Stage 2 `generate_code`.

## 8. Release the runtime

Compute units bill on GPU-connected wall time and closing the tab does not
disconnect. Run this once section 6 has written the results to Drive.

In [ ]:
from google.colab import runtime

runtime.unassign()